In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, pairwise_distances
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
import torch.optim as optim
import random
import joblib


In [5]:
#get version info
import importlib.metadata
import sys

# List of main packages you want versions for
packages = [
    'numpy', 'pandas', 'scikit-learn', 'torch','joblib'
]

for pkg in packages:
    try:
        # Try to get version from importlib.metadata
        version = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        try:
            # fallback: get version from __version__ if already imported
            mod = __import__(pkg)
            version = getattr(mod, '__version__', 'unknown')
        except Exception:
            version = 'unknown'
    print(f"{pkg}: {version}")

# Optional: Python version
print(f"Python: {sys.version}")

numpy: 1.26.4
pandas: 2.2.2
scikit-learn: 1.5.2
torch: 1.13.1
joblib: 1.4.2
Python: 3.9.2 (v3.9.2:1a79785e3e, Feb 19 2021, 09:06:10) 
[Clang 6.0 (clang-600.0.57)]


In [8]:
#Import Data and deal with missing values
#test_data = pd.read_csv("house-prices-advanced-regression-techniques/test.csv")
train_data = pd.read_csv("../house-prices-advanced-regression-techniques/train.csv")

#test_data.info() #check data
print("train set size:", len(train_data))

train_missing = train_data.isnull().sum()
print(train_missing[train_missing != 0].sort_values(ascending=False))
print()

#print("test set size: ", len(test_data))
#test_missing = test_data.isnull().sum()
#print(test_missing[test_missing != 0].sort_values(ascending=False))
#print(test_data['MSZoning'].unique())
#print(test_data[test_data['MSZoning'].isnull()])

#replace nan with none where nan is semantically meaningful
nan_catagory_columns = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType',
                        'FireplaceQu', 'GarageCond', 'GarageFinish', 'GarageQual',
                        'BsmtCond', 'BsmtExposure', 'BsmtQual', 'BsmtFinType1',
                        'BsmtFinType2']


missing_cols = [col for col in nan_catagory_columns if col not in train_data.columns]
print("missing columns: ", missing_cols)

#nan_catagory_columns_test = [col for col in nan_catagory_columns if col in test_data.columns]
nan_catagory_columns_train = [col for col in nan_catagory_columns if col in train_data.columns]

##test_data[nan_catagory_columns_test] = test_data[nan_catagory_columns_test].fillna('None')
train_data[nan_catagory_columns_train] = train_data[nan_catagory_columns_train].fillna('None')

#replace nan with 0 where needed
nan_zero_columns = ['GarageYrBlt', 'MasVnrArea']
#test_data[nan_zero_columns] = test_data[nan_zero_columns].fillna(0)
train_data[nan_zero_columns] = train_data[nan_zero_columns].fillna(0)

#replace nan with mode in all other catagorical columns
categorical_cols = train_data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    #test_data[col].fillna(test_data[col].mode()[0], inplace=True)
    train_data[col].fillna(train_data[col].mode()[0], inplace=True)

#replace nan with median in all other numnerical columns
numeric_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    #test_data[col].fillna(test_data[col].median(), inplace=True)
    train_data[col].fillna(train_data[col].median(), inplace=True)


#check for remaining NANs
nan_count = train_data.isna().sum()
pd.set_option('display.max_rows', None)
print("nan columns:")
print(nan_count)









train set size: 1460
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64

missing columns:  []
nan columns:
Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
Street           0
Alley            0
LotShape         0
LandContour      0
Utilities        0
LotConfig        0
LandSlope        0
Neighborhood     0
Condition1       0
Condition2       0
BldgType         0
HouseStyle       0
OverallQual      0
OverallCond      0
YearBuilt        0
YearRemodAdd     0
RoofStyle        0
RoofMatl         0
Exterior1st      0
Exterior2nd      0
MasVnrType       0
MasVnrArea       0
ExterQual        0


/var/folders/r7/ydwb86214mb50pm6tdk2x75m0000gn/T/ipykernel_1127/3268840170.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data[col].fillna(train_data[col].mode()[0], inplace=True)
/var/folders/r7/ydwb86214mb50pm6tdk2x75m0000gn/T/ipykernel_1127/3268840170.py:49: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

In [9]:
#Preprocess Data

#One hot encode catagorical features
preen_categorical_cols = train_data.select_dtypes(include=['object']).columns
train_data = pd.get_dummies(train_data, drop_first=True)
#test_data = pd.get_dummies(test_data, drop_first=True)
ohe_map = {}
for col in preen_categorical_cols:
    ohe_map[col] = [c for c in train_data.columns if c.startswith(col + "_")]


#Scale numeric features
scaler = StandardScaler()
numeric_cols = train_data.select_dtypes(include=['int64','float64']).columns.drop("SalePrice") #get all columns minus y
train_data[numeric_cols] = scaler.fit_transform(train_data[numeric_cols])
#test_data[numeric_cols] = scaler.transform(test_data[numeric_cols])


#fix skewed values 
skewed = train_data[numeric_cols].apply(lambda x: x.skew()).sort_values(ascending=False)
skewed_features = skewed[abs(skewed) > 0.75].index
pt = PowerTransformer(method='yeo-johnson')
train_data[skewed_features] = pt.fit_transform(train_data[skewed_features])
#test_data[skewed_features] = pt.fit_transform(test_data[skewed_features])

#check for remaining NANs
nan_count = train_data.isna().sum()
pd.set_option('display.max_rows', None)
print("nan columns:")
print(nan_count)

#drop negative skewed values (not needed anymore)
#neg_skewed_features = skewed[skewed < -0.75].index
#train_data.drop(columns=list(neg_skewed_features), inplace=True)
#test_data.drop(columns=list(neg_skewed_features), inplace=True)


nan columns:
Id                       0
MSSubClass               0
LotFrontage              0
LotArea                  0
OverallQual              0
OverallCond              0
YearBuilt                0
YearRemodAdd             0
MasVnrArea               0
BsmtFinSF1               0
BsmtFinSF2               0
BsmtUnfSF                0
TotalBsmtSF              0
1stFlrSF                 0
2ndFlrSF                 0
LowQualFinSF             0
GrLivArea                0
BsmtFullBath             0
BsmtHalfBath             0
FullBath                 0
HalfBath                 0
BedroomAbvGr             0
KitchenAbvGr             0
TotRmsAbvGrd             0
Fireplaces               0
GarageYrBlt              0
GarageCars               0
GarageArea               0
WoodDeckSF               0
OpenPorchSF              0
EnclosedPorch            0
3SsnPorch                0
ScreenPorch              0
PoolArea                 0
MiscVal                  0
MoSold                   0
YrSold         

In [10]:
#Feature Selection

#Get feature correlation (not used can remove)
numeric_cols = train_data.select_dtypes(include=['int64','float64']).columns.drop("SalePrice") #get all columns minus y
corr = train_data[numeric_cols].corrwith(train_data['SalePrice']).abs().sort_values(ascending=False)
#print(corr.head(20))
#print()

#Groupfeature importance
x = train_data.drop('SalePrice', axis=1)
y = np.log1p(train_data['SalePrice'])  # log-transform target
rf = RandomForestRegressor(n_estimators=100) #get a random forest
rf.fit(x, y)
importances = pd.Series(rf.feature_importances_, index=x.columns).sort_values(ascending=False)
def group_importances(importances, ohe_map):
    grouped = {}
    for feat, dummies in ohe_map.items():
        grouped[feat] = importances[dummies].sum()  # sum all dummies
    # Add numeric features directly
    for feat in x.columns:
        if feat not in sum(ohe_map.values(), []):  # not a dummy
            grouped[feat] = importances[feat]
    return pd.Series(grouped).sort_values(ascending=False)

grouped_importances = group_importances(importances, ohe_map)
print("Grouped feature importance:")
print(grouped_importances.head(20))
print()



#scikit learn RFE (not used can remove)
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=20)
rfe.fit(x, y)
rfe_selected_features = x.columns[rfe.support_]
#print("scikit learn RFE:")
#print(rfe_selected_features)

#mutual information regression
mi = mutual_info_regression(x, y)
mi_series = pd.Series(mi, index=x.columns).sort_values(ascending=False)
grouped_mi = group_importances(mi_series, ohe_map)
print("Grouped mutual information regression")
print(grouped_mi.head(20))



#get final features based on group feature importance and mutual information regression
selected_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
                     'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'TotRmsAbvGrd',
                     'FullBath', 'YearRemodAdd', 'Neighborhood', 'CentralAir',
                     'LotArea', 'OverallCond', '2ndFlrSF', 'MSZoning', 
                     'KitchenQual', 'ExterQual', 'BsmtQual', 'FireplaceQu',
                     'SalePrice']


#add one hot encoded features
selected_features = [
    col for feat in selected_features
    for col in ([feat] if feat in train_data.columns 
                else [c for c in train_data.columns if c.startswith(feat + "_")])
]

print("Missing columns: ", [col for col in selected_features if col not in train_data.columns])

train_data = train_data.loc[:, train_data.columns.intersection(selected_features)]

print("Final features")
for col in train_data.columns:
    print(col)


print("training data final columns")
print(train_data.columns)



Grouped feature importance:
OverallQual     0.549391
GrLivArea       0.107721
GarageCars      0.051523
TotalBsmtSF     0.042398
1stFlrSF        0.026080
GarageArea      0.023790
BsmtFinSF1      0.020284
YearBuilt       0.013151
LotArea         0.012832
CentralAir      0.012103
OverallCond     0.011215
YearRemodAdd    0.007754
GarageYrBlt     0.007425
BsmtUnfSF       0.005735
MSZoning        0.005619
2ndFlrSF        0.005582
LotFrontage     0.005491
GarageType      0.005316
Fireplaces      0.005133
Neighborhood    0.005095
dtype: float64

Grouped mutual information regression
OverallQual     0.574610
Neighborhood    0.555114
ExterQual       0.497063
GrLivArea       0.479328
KitchenQual     0.425116
GarageArea      0.363598
BsmtQual        0.363592
TotalBsmtSF     0.362216
GarageCars      0.360431
YearBuilt       0.356313
Foundation      0.303746
1stFlrSF        0.296961
FireplaceQu     0.295803
GarageYrBlt     0.293108
GarageFinish    0.277293
MSSubClass      0.269133
FullBath        0.

In [11]:
#Feature Engineering
train_data['TotalSF'] = train_data['TotalBsmtSF'] + train_data['1stFlrSF'] + train_data['2ndFlrSF']
train_data['CurrHouseAge'] = train_data['YearRemodAdd'] - train_data['YearBuilt']
train_data['GarageSizePerCar'] = train_data['GarageArea']/train_data['GarageCars']
train_data['BathPerRoom'] = train_data['FullBath']/train_data['TotRmsAbvGrd']
train_data['LotRatio'] = train_data['GrLivArea']/train_data['LotArea']
train_data['GarageLotRatio'] = train_data['GarageArea']/train_data['LotArea']

In [12]:
#split into x and y datasets
#all data below only uses data from train.csv

X = train_data.drop('SalePrice', axis=1)
y = train_data['SalePrice']  # or log-transformed if needed

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Check y skew
skew_value = train_data['SalePrice'].skew()
print(f"SalePrice skew: {skew_value:.2f}")

#y is right skewed so log transform
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

SalePrice skew: 1.88


In [15]:
#Linear regression
#train linear regression
lr = LinearRegression()
lr.fit(X_train, y_train_log)

print(X_train.columns)

#make predictions
y_pred_log_linreg = lr.predict(X_val)

#evaluate model in log space
rmse_log_linreg = np.sqrt(mean_squared_error(y_val_log, y_pred_log_linreg))
r2_log_linreg = r2_score(y_val_log, y_pred_log_linreg)
print(f"Linear regression RMSE log space:  {rmse_log_linreg:.4f}")
print(f"Linear regression r2 score log space: {r2_log_linreg:.4f}")

#evaluate model in non log space
y_pred_linreg = np.expm1(y_pred_log_linreg)
rmse_linreg = np.sqrt(mean_squared_error(y_val, y_pred_linreg))
r2_linreg = r2_score(y_val, y_pred_linreg)
print(f"Linear regression RMSE: ${rmse_linreg:.2f}")
print(f"Linear regression r2 score: {r2_linreg:.4f}")


Index(['LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'FullBath',
       'TotRmsAbvGrd', 'GarageCars', 'GarageArea', 'MSZoning_FV',
       'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM', 'Neighborhood_Blueste',
       'Neighborhood_BrDale', 'Neighborhood_BrkSide', 'Neighborhood_ClearCr',
       'Neighborhood_CollgCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards',
       'Neighborhood_Gilbert', 'Neighborhood_IDOTRR', 'Neighborhood_MeadowV',
       'Neighborhood_Mitchel', 'Neighborhood_NAmes', 'Neighborhood_NPkVill',
       'Neighborhood_NWAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt',
       'Neighborhood_OldTown', 'Neighborhood_SWISU', 'Neighborhood_Sawyer',
       'Neighborhood_SawyerW', 'Neighborhood_Somerst', 'Neighborhood_StoneBr',
       'Neighborhood_Timber', 'Neighborhood_Veenker', 'ExterQual_Fa',
       'ExterQual_Gd', 'ExterQual_TA', 'BsmtQual_Fa', 'BsmtQual_Gd',
       'BsmtQual_None', 'B

In [17]:
#Ridge Regression
#train ridge regression
ridge = Ridge(alpha=1) #alphas 0.01, 0.1, 1, 10, 100 tested. Alpha = 1 yields best result
ridge.fit(X_train, y_train_log)

#log space prediction
y_pred_log_ridreg = ridge.predict(X_val)

#convert to non log space
y_pred_ridreg = np.expm1(y_pred_log_ridreg)

#evaluate model
rmse_ridreg = np.sqrt(mean_squared_error(y_val, y_pred_ridreg))
r2_ridreg = r2_score(y_val, y_pred_ridreg)
print(f"Linear regression RMSE: ${rmse_ridreg:.2f}")
print(f"Linear regression r2 score: {r2_ridreg:.4f}")

Linear regression RMSE: $28043.71
Linear regression r2 score: 0.8975


In [19]:
#Lasso Regression
#train lasso regression
lasso = Lasso(alpha=0.0001, max_iter=10000) #alphas 0.0001, 0.001, 0.01, 0.1, 1, 10 tested. Alpha = 0.0001 yields best result
lasso.fit(X_train, y_train_log)

#log space prediction
y_pred_log_lasso = lasso.predict(X_val)

#convert to non log space
y_pred_lasso = np.expm1(y_pred_log_lasso)

#evaluate model
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))
r2_lasso = r2_score(y_val, y_pred_lasso)
print(f"Linear regression RMSE: ${rmse_lasso:.2f}")
print(f"Linear regression r2 score: {r2_lasso:.4f}")

Linear regression RMSE: $28064.33
Linear regression r2 score: 0.8973


In [21]:
#ElasticNet
#alphas 0.0001, 0.001, 0.01, 0.1, 1, 10 tested. Alpha = 0.0001 yields best result
#l1_ratios 0.1, 0.3, 0.5, 0.7, 0.9 tested. l1_ratio = 0.7 yields best result (70% Lasso)
elastic_net = ElasticNet(alpha=0.0001, l1_ratio = 0.7, max_iter=10000) 
elastic_net.fit(X_train, y_train_log)

#log space prediction
y_pred_log_elnet = elastic_net.predict(X_val)

#convert to non log space
y_pred_elnet = np.expm1(y_pred_log_elnet)

#evaluate model
rmse_elnet = np.sqrt(mean_squared_error(y_val, y_pred_elnet))
r2_elnet = r2_score(y_val, y_pred_elnet)
print(f"Linear regression RMSE: ${rmse_elnet:.2f}")
print(f"Linear regression r2 score: {r2_elnet:.4f}")

Linear regression RMSE: $27985.86
Linear regression r2 score: 0.8979


In [23]:
#Decision Tree Regressor
#train decision tree
#max_depth, min_samples_split multiple values tested
dt_reg = DecisionTreeRegressor(max_depth = 7, min_samples_split=2, random_state=42)
dt_reg.fit(X_train, y_train)

#Predict
y_pred_dt = dt_reg.predict(X_val)

#evaluate
rmse_dt = np.sqrt(mean_squared_error(y_val, y_pred_dt))
r2_dt = r2_score(y_val, y_pred_dt)
print(f"Decision Tree RMSE: ${rmse_dt:.2f}")
print(f"Decision Tree R²: {r2_dt:.4f}")

#plot tree
#plt.figure(figsize=(20,10))
#plot_tree(
#    dt_reg,
#    feature_names=X_train.columns,
#    filled=True,
#    rounded=True,
#    fontsize=10
#)
#plt.show()


Decision Tree RMSE: $38346.69
Decision Tree R²: 0.8083


In [25]:
#Random Forest Regressor

#get best hyperparameters
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [5, 7, 10, None],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2']
# }
# rf = RandomForestRegressor(random_state=42)
# grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error')
# grid_search.fit(X_train, y_train)
# print("Best params:", grid_search.best_params_)


#train random forest regressor with result of hyperparameter search
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    max_features = 'sqrt',
    random_state=42,
    min_samples_split = 2
)
rf_reg.fit(X_train, y_train)

#predict
y_pred_rf = rf_reg.predict(X_val)

#evaluate
rmse_rf = np.sqrt(mean_squared_error(y_val,y_pred_rf))
r2_rf = r2_score(y_val, y_pred_rf)
print(f"Random Forest RMSE: ${rmse_rf:.2f}")
print(f"Random Forest R²: {r2_rf:.4f}")

Random Forest RMSE: $28947.44
Random Forest R²: 0.8908


In [27]:
#Gradient Boosting Regressor

#Search for best hyperparamters
# param_grid = {
#     'n_estimators': [100, 200, 300],         
#     'learning_rate': [0.01, 0.05, 0.1],     
#     'max_depth': [3, 5, 7],                  
#     'min_samples_split': [2, 5, 10],         
#     'min_samples_leaf': [1, 2, 4]            
# }
# gbr = GradientBoostingRegressor(random_state=42)
# grid_search = GridSearchCV( estimator=gbr, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
# grid_search.fit(X_train, y_train)
# print("Best params:", grid_search.best_params_)
#Best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

#train gradient boosting regressor
gb_reg = GradientBoostingRegressor(
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=300,
    random_state=42
)
gb_reg.fit(X_train, y_train)

#predict
y_pred_gb = gb_reg.predict(X_val)

#evaluate
rmse_gb = np.sqrt(mean_squared_error(y_val, y_pred_gb))
r2_gb = r2_score(y_val, y_pred_gb)
print(f"Random Forest RMSE: ${rmse_gb:.2f}")
print(f"Random Forest R²: {r2_gb:.4f}")


Random Forest RMSE: $27767.47
Random Forest R²: 0.8995


In [28]:
#Stacking Regressor (Linear and tree based models)

#base model training set predictions in log space
train_predictions_log = pd.DataFrame({
    'linreg': lr.predict(X_train),
    'ridge': ridge.predict(X_train),
    'lasso': lasso.predict(X_train),
    'ElasticNet': elastic_net.predict(X_train),
    'dt_reg': np.log1p(dt_reg.predict(X_train)),
    'rf_reg': np.log1p(rf_reg.predict(X_train)),
    'gb_reg': np.log1p(gb_reg.predict(X_train))
})

#base model validation set predictions in log space
val_predictions_log = pd.DataFrame({
    'linreg': lr.predict(X_val),
    'ridge': ridge.predict(X_val),
    'lasso': lasso.predict(X_val),
    'ElasticNet': elastic_net.predict(X_val),
    'dt_reg': np.log1p(dt_reg.predict(X_val)),
    'rf_reg': np.log1p(rf_reg.predict(X_val)),
    'gb_reg': np.log1p(gb_reg.predict(X_val))
})

#train meta model in log space
meta_model = LinearRegression()
meta_model.fit(train_predictions_log, y_train_log)

#get stacked prediction in log space space
y_pred_stack_log = meta_model.predict(val_predictions_log)

# print("Stacked predictions (log space):")
# print(pd.Series(y_pred_stack_log).describe())

#convert to dollar space
y_pred_stack = np.expm1(y_pred_stack_log)


#evaluate
rmse_stack = np.sqrt(mean_squared_error(y_val, y_pred_stack))
r2_stack = r2_score(y_val, y_pred_stack)
print(f"Random Forest RMSE: ${rmse_stack:.2f}")
print(f"Random Forest R²: {r2_stack:.4f}")


Random Forest RMSE: $27535.19
Random Forest R²: 0.9012


In [30]:
#Voting Regressor

#get validation predictions in dollar space (models already trained so dont need train)
val_predictions = val_predictions_log.apply(np.expm1)


#get average
y_pred_vote = val_predictions.mean(axis=1)

#evaluate
rmse_vote = np.sqrt(mean_squared_error(y_val, y_pred_vote))
r2_vote = r2_score(y_val, y_pred_vote)
print(f"Random Forest RMSE: ${rmse_vote:.2f}")
print(f"Random Forest R²: {r2_vote:.4f}")



Random Forest RMSE: $27352.83
Random Forest R²: 0.9025


In [31]:
#Weighted Voting Regressor

#get weights based on best performing models
vote_weights = np.array([2, 2, 2, 2, 1, 1, 1])
vote_weights = vote_weights/vote_weights.sum()

#get weighted predictions
y_pred_vote_weighted = (val_predictions*vote_weights).sum(axis=1)

#evaluate
rmse_vote_weighted = np.sqrt(mean_squared_error(y_val, y_pred_vote_weighted))
r2_vote_weighted = r2_score(y_val, y_pred_vote_weighted)
print(f"Random Forest RMSE: ${rmse_vote_weighted:.2f}")
print(f"Random Forest R²: {r2_vote_weighted:.4f}")

Random Forest RMSE: $27358.12
Random Forest R²: 0.9024


In [32]:
#Prepare data for torch


print(X_train.columns)

#Get data in torch tensors
X_train_t = torch.tensor(X_train.values.astype(np.float32))
X_val_t = torch.tensor(X_val.values.astype(np.float32))
y_train_t_log = torch.tensor(y_train_log.values.astype(np.float32)).view(-1, 1)
y_val_t_log = torch.tensor(y_val_log.values.astype(np.float32)).view(-1, 1)

#normalize target
y_train_t_log_mean = y_train_t_log.mean()
y_train_t_log_std = y_train_t_log.std()
y_train_t_log_norm = (y_train_t_log - y_train_t_log_mean)/y_train_t_log_std

#set seed for consistent results
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

Index(['LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'FullBath',
       'TotRmsAbvGrd', 'GarageCars', 'GarageArea', 'MSZoning_FV',
       'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM', 'Neighborhood_Blueste',
       'Neighborhood_BrDale', 'Neighborhood_BrkSide', 'Neighborhood_ClearCr',
       'Neighborhood_CollgCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards',
       'Neighborhood_Gilbert', 'Neighborhood_IDOTRR', 'Neighborhood_MeadowV',
       'Neighborhood_Mitchel', 'Neighborhood_NAmes', 'Neighborhood_NPkVill',
       'Neighborhood_NWAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt',
       'Neighborhood_OldTown', 'Neighborhood_SWISU', 'Neighborhood_Sawyer',
       'Neighborhood_SawyerW', 'Neighborhood_Somerst', 'Neighborhood_StoneBr',
       'Neighborhood_Timber', 'Neighborhood_Veenker', 'ExterQual_Fa',
       'ExterQual_Gd', 'ExterQual_TA', 'BsmtQual_Fa', 'BsmtQual_Gd',
       'BsmtQual_None', 'B

In [37]:
#Multilayer Perceptron


#hyperparameters
per_learning_rate = 0.001
epochs = 500
# adam optimizer

#define perceptron
class Perceptron(nn.Module):
    
    #define layers of the network
    def __init__(self, n_features):
        super().__init__() #Call parent instructor
        self.fc1 = nn.Linear(n_features, n_features*2) #input layer
        self.act1 = nn.LeakyReLU()
        self.fc2 = nn.Linear(n_features*2, n_features) #hidden layer
        self.act2 = nn.LeakyReLU()
        self.out = nn.Linear(n_features, 1) #output layer

    #define forward pass rules
    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        x = self.out(x)
        return x #linear activation function
    
#Initialize model, loss, and optimizer
model_per = Perceptron(X_train.shape[1]) #pass constructor # of features
criterion_per = nn.MSELoss()
optimizer_per = optim.Adam(model_per.parameters(), lr=per_learning_rate, weight_decay=1e-5)

#Train model
for epoch in range(epochs):
    model_per.train() #put model in training mode
    optimizer_per.zero_grad() #clear previous gradient
    outputs = model_per(X_train_t) #forward pass
    loss = criterion_per(outputs, y_train_t_log_norm) #get loss
    loss.backward() #get gradients with back pass
    optimizer_per.step() #update model using gradients
    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


#Evaluate
model_per.eval()
with torch.no_grad(): #get predictions without gradients
    pred_per_log = (model_per(X_val_t)*y_train_t_log_std+y_train_t_log_mean).numpy().flatten()
    pred_per = np.expm1(pred_per_log)

rmse_per = np.sqrt(mean_squared_error(y_val, pred_per))
r2_per = r2_score(y_val, pred_per)
print(f"Perceptron RMSE: ${rmse_per:.2f}")
print(f"Perceptron R²: {r2_per:.4f}")

Epoch [50/500], Loss: 0.1949
Epoch [100/500], Loss: 0.1199
Epoch [150/500], Loss: 0.0910
Epoch [200/500], Loss: 0.0746
Epoch [250/500], Loss: 0.0648
Epoch [300/500], Loss: 0.0567
Epoch [350/500], Loss: 0.0499
Epoch [400/500], Loss: 0.0430
Epoch [450/500], Loss: 0.0366
Epoch [500/500], Loss: 0.0296
Perceptron RMSE: $31605.20
Perceptron R²: 0.8698


In [38]:
#Radias Basis Function Network

#hyperparameters
num_hidden_neurons = 100
spread = 5
rbf_learning_rate = 0.01
epochs = 500
# adam optimizer


#Define Custom RBF Layer
class RBFLayer(nn.Module):
    def __init__(self, in_features, out_features, spread, X_train):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        

        #initialize centres with k means
        kmeans = KMeans(n_clusters=out_features, random_state=42).fit(X_train.numpy())
        self.centers = nn.Parameter(torch.tensor(kmeans.cluster_centers_, dtype=torch.float32))
        
        
        #Initlizlize spread as mean distance between k-means centers
        center_distances = pairwise_distances(kmeans.cluster_centers_)
        self.spread = nn.Parameter(torch.tensor([np.mean(center_distances)], dtype=torch.float32))

    def forward(self, x):
    
        #get input_data to match center data dimension
        x = x.unsqueeze(1).expand(-1, self.out_features, -1)

        #get centers to match input_data
        c = self.centers.unsqueeze(0)

        #return gaussian function
        dist = torch.norm(x - c, dim=2)
        return torch.exp(-self.spread * dist ** 2)
    
#Define RBF Network
class RBFNet(nn.Module):
    def __init__(self, n_features, hidden_neurons, spread, X_train):
        super().__init__()

        #make RBF layer
        self.rbflayer = RBFLayer(n_features, hidden_neurons, spread, X_train)

        #make linear output
        self.linear = nn.Linear(hidden_neurons, 1)

    def forward(self, x):
        x = self.rbflayer(x)
        x = self.linear(x)
        return x

#Initlize model
n_features = X_train.shape[1]
model_rbf = RBFNet(n_features, num_hidden_neurons, spread, X_train_t)

#Loss and optimizer
criterion_rbf = nn.MSELoss()
optimizer_rbf = optim.Adam(model_rbf.parameters(), lr=rbf_learning_rate)

#Train Model
for epoch in range(epochs):
    model_rbf.train() #put model in training mode
    optimizer_rbf.zero_grad() #clear previous gradient
    outputs = model_rbf(X_train_t) #forward pass
    loss = criterion_rbf(outputs, y_train_t_log_norm) #get loss
    loss.backward() #get gradients with back pass
    optimizer_rbf.step() #update model using gradients
    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

#Evaluate
model_rbf.eval()
with torch.no_grad(): #get predictions without gradients
    pred_rbf_log = (model_rbf(X_val_t)*y_train_t_log_std+y_train_t_log_mean).numpy().flatten()
    pred_rbf = np.expm1(pred_rbf_log)

rmse_rbf = np.sqrt(mean_squared_error(y_val, pred_rbf))
r2_rbf = r2_score(y_val, pred_rbf)
print(f"RBF RMSE: ${rmse_rbf:.2f}")
print(f"RBF R²: {r2_rbf:.4f}")

/Users/nilecochen/venvs/rpi3-env/lib/python3.9/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Epoch [50/500], Loss: 0.9877
Epoch [100/500], Loss: 0.9809
Epoch [150/500], Loss: 0.9783
Epoch [200/500], Loss: 0.9776
Epoch [250/500], Loss: 0.9774
Epoch [300/500], Loss: 0.9773
Epoch [350/500], Loss: 0.9773
Epoch [400/500], Loss: 0.9773
Epoch [450/500], Loss: 0.9773
Epoch [500/500], Loss: 0.9773
RBF RMSE: $88257.26
RBF R²: -0.0155


In [39]:
#Export Models


# --- 1. Create TorchScript versions of the PyTorch models ---
mlp_scripted = torch.jit.script(model_per)  # MLP
rbf_scripted = torch.jit.script(model_rbf)  # RBF

# TorchScript models
mlp_scripted.save('mlp_model_scripted.pt')
rbf_scripted.save('rbf_model_scripted.pt')



#make skikit learn dictionary
sklearn_models = {
    'linear': lr,
    'ridge': ridge,
    'lasso': lasso,
    'elastic_net': elastic_net,
    'decision_tree': dt_reg,
    'random_forest': rf_reg,
    'gradient_boosting': gb_reg,
}

#make torch dictionary
torch_models = {
    'Perceptron_scripted': 'mlp_model_scripted.pt',
    'rbf_scripted': 'rbf_model_scripted.pt'
}

#organize metrics
metrics = {
    'rmse': {
        'linear': rmse_linreg,
        'ridge': rmse_ridreg,
        'lasso': rmse_lasso,
        'elastic_net': rmse_elnet,
        'decision_tree': rmse_dt,
        'random_forest': rmse_rf,
        'gradient_boosting': rmse_gb,
        'MLP': rmse_per,
        'RBF': rmse_rbf
    },
    'r2': {
        'linear': r2_linreg,
        'ridge': r2_ridreg,
        'lasso': r2_lasso,
        'elastic_net': r2_elnet,
        'decision_tree': r2_dt,
        'random_forest': r2_rf,
        'gradient_boosting': r2_gb,
        'MLP': r2_per,
        'RBF': r2_rbf
    }
}

#organize preprocessing tools
preprocessing = {
    'scaler': scaler,
    'power_transformer': pt,
    'ohe_map': ohe_map,
    'y_train_t_log_mean': y_train_t_log_mean,
    'y_train_t_log_std': y_train_t_log_std
}

#make final complete export dictionary
export_dict = {
    'sklearn_models': sklearn_models,
    'torch_models': torch_models,
    'metrics': metrics,
    'preprocessing': preprocessing
}

joblib.dump(export_dict,'all_models_export.pkl')



['all_models_export.pkl']

In [ ]:
#Test exported models